# Benchmark

Scores every model in the project on the test split and the two outside test sets, measures CPU latency, and builds the tables and charts.

`Runtime > Change runtime type > GPU`, then `Runtime > Run all`. Code and data come from the team repo. The results are copied to Google Drive (`MyDrive/mayi_benchmark/`) after every step, so a disconnect loses at most one step.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python --version

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/mayi_benchmark'
!mkdir -p $OUT

In [ ]:
%cd /content
!test -d nlp-project-1 || git clone -q https://github.com/ata-sesli/nlp-project-1
%cd /content/nlp-project-1
!git log --oneline -1
!pip install -q -r benchmark/requirements.txt

## Settings
`LIMIT = 0` scores every row. For a quick check that everything works, set it to `100` and run all cells first.

In [ ]:
LIMIT = 0

def save():
    """Copy what exists so far to Drive."""
    !mkdir -p benchmark/results benchmark/predictions
    !cp -r benchmark/results benchmark/predictions $OUT/
    print('saved to', OUT)

## 1) TF-IDF + Logistic Regression
The trained models are not in the repo, so they are trained here first.

In [ ]:
!python baseline/src/baseline.py --input input_a
!python baseline/src/baseline.py --input input_b

In [ ]:
!python benchmark/score_baseline.py --limit $LIMIT
save()

## 2) DistilBERT
The fine-tuned models are downloaded from the Hugging Face Hub.

In [ ]:
!python benchmark/score_distilbert.py --limit $LIMIT
save()

## 3) Julia 1 (zero-shot)
A general decision model, not trained on our data. It scores one row at a time, so this is the slowest step. If it fails or takes too long, skip it: the benchmark below works without it.

In [ ]:
!python -c "from huggingface_hub import snapshot_download; snapshot_download('SupersonicLabs/Julia-1', local_dir='Julia-1')"
!pip install -q -e ./Julia-1

In [ ]:
!python benchmark/score_julia.py --model-dir Julia-1 --limit $LIMIT
save()

## 4) Tables and charts

In [ ]:
!python benchmark/run_benchmark.py
save()

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(open('benchmark/results/benchmark.md').read()))

In [ ]:
import glob
for png in sorted(glob.glob('benchmark/results/*.png')):
    print(png)
    display(Image(png, width=700))

## 5) Download
The same files are already on Drive. This also downloads `results/` as a zip, to commit to the repo under `benchmark/results/`.

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('/content/benchmark_results', 'zip', 'benchmark/results')
files.download('/content/benchmark_results.zip')